In [1]:
from google.colab import files
import pandas as pd
import io
from IPython.display import Image, display

In [2]:
 uploaded = files.upload()

Saving europe.csv to europe.csv


In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import altair as alt

# Load the dataset
df = pd.read_csv(io.BytesIO(uploaded['europe.csv']))

# Select numeric columns
numerical_cols = df.select_dtypes(include=np.number).columns

# Standardize
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[numerical_cols])

# PCA
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(scaled_data)
pca_loadings = pca.components_.T

# PCA scores DataFrame
scores_df = pd.DataFrame(pca_scores, columns=['PC1', 'PC2'])
scores_df['Country'] = df['Country']

# Add the original (unscaled) values as columns for tooltips
for col in numerical_cols:
    scores_df[col] = df[col].values

# PCA loadings DataFrame
loadings_df = pd.DataFrame(pca_loadings, columns=['PC1', 'PC2'])
loadings_df['Feature'] = numerical_cols
scale = 2
loadings_df['PC1_scaled'] = loadings_df['PC1'] * scale
loadings_df['PC2_scaled'] = loadings_df['PC2'] * scale
loadings_df['zero'] = 0

# PCA scores as points with full variable tooltips
points = alt.Chart(scores_df).mark_circle(size=60).encode(
    x=alt.X('PC1', title='PC1 x 2'),
    y=alt.Y('PC2', title='PC2 x 2'),
    tooltip=['Country'] + list(numerical_cols)
)

# Loadings as arrows
arrows = alt.Chart(loadings_df).mark_rule(color='red').encode(
    x='zero',
    y='zero',
    x2='PC1_scaled',
    y2='PC2_scaled'
)

# Arrow labels
arrow_labels = alt.Chart(loadings_df).mark_text(
    color='red',
    fontWeight='bold',
    dx=5,
    dy=-5
).encode(
    x='PC1_scaled',
    y='PC2_scaled',
    text='Feature'
)

# Combine and show
biplot = (points + arrows + arrow_labels).properties(
    width=700,
    height=600,
    title='PCA Biplot (Scores and Loadings)'
)

biplot

alt.LayerChart(...)

Se toman solo las variables numéricas de nuestros datos (todo menos el nombre de país) y se estandarizan los datos. Luego se calculan los primeros 2 componentes de PCA.

Estos componentes representarán las direcciones de mayor variabilidad de nuestros datos, con el PC1 representando más variabilidad que PC2.

Finalmente se grafica PC1 vs PC2 donde cada país es representado por un punto según su valor en cada componente. Además se agregaron vectores para cada variable donde la punta del vector se define por la carga del PC1 y PC2 para esa variable (con una escala de x2 para hacer más notables las flechas).

In [7]:
# Mapeo de desplazamientos por país
offsets = {
    'Germany': {'dx': 10, 'dy': -12},
    'Belgium': {'dx': -20, 'dy': 15},
    'Denmark': {'dx': 25, 'dy': 3},
    'Sweden': {'dx': 10, 'dy': 10},
    'Finland': {'dx': 25, 'dy': -1},
    'Czech Republic': {'dx': 45, 'dy': -1},
    'Netherlands': {'dx': 0, 'dy': -10},
    'Switzerland': {'dx': 0, 'dy': -10},
    'Luxembourg': {'dx': 0, 'dy': -10},
    'Lithuania': {'dx': 30, 'dy': 0},
    'United Kingdom': {'dx': 0, 'dy': 10},
}

# Agregamos columnas auxiliares de desplazamiento
scores_df['dx'] = scores_df['Country'].map(lambda c: offsets.get(c, {'dx': 25})['dx'])
scores_df['dy'] = scores_df['Country'].map(lambda c: offsets.get(c, {'dy': 0})['dy'])

# Etiquetas con desplazamiento variable usando transform_calculate
country_labels = alt.Chart(scores_df).transform_calculate(
    dx="datum.dx",
    dy="datum.dy"
).mark_text(
    align='left',
    baseline='middle',
    fontSize=7,
    color='black'
).encode(
    x='PC1',
    y='PC2',
    text='Country'
).properties()

# Aplica desplazamientos
country_labels = country_labels.encode(
    x=alt.X('PC1:Q'),
    y=alt.Y('PC2:Q')
).mark_text(
    dx=alt.ExprRef(expr='datum.dx'),
    dy=alt.ExprRef(expr='datum.dy')
)

# Biplot final
biplot = (points + arrows + arrow_labels + country_labels).properties(
    width=700,
    height=600,
    title='PCA Biplot (Scores and Loadings)'
)

biplot


alt.LayerChart(...)

Observando el gráfico de arriba podemos separar a las variables en 4 grupos según el signo de su carga en PC1 y PC2. Los grupos quedan:
- Military, Unemployment
- Life expectancy, population growth
- GDP
- Area, Inflation

Esto nos indica que para nuestros datos hay una relación directa entre cantidad de ejército y desempleo, área e inflación, y expectativa de vida, crecimiento poblacional y GDP.

In [8]:
display(Image('ben-diagram.png'))

FileNotFoundError: No such file or directory: 'ben-diagram.png'

FileNotFoundError: No such file or directory: 'ben-diagram.png'

<IPython.core.display.Image object>

Con la siguiente tabla se puede explicar un poco mejor lo de los vectores de las variables. Por ejemplo para `Area` graficamos una línea desde el `(0, 0)` hasta el `2 * (-0.124874, -0.172872)` (el `2*` por el escalamiento usado).

In [9]:
loadings_df

,PC1,PC2,Feature,PC1_scaled,PC2_scaled,zero
0,-0.124874,-0.172872,Area,-0.249748,-0.345744,0
1,0.500506,-0.130140,GDP,1.001012,-0.260279,0
2,-0.406518,-0.369657,Inflation,-0.813036,-0.739314,0
3,0.482873,0.265248,Life.expect,0.965747,0.530496,0
4,-0.188112,0.658267,Military,-0.376223,1.316534,0
5,0.475704,0.082622,Pop.growth,0.951407,0.165244,0
6,-0.271656,0.553204,Unemployment,-0.543312,1.106407,0


Ahora calculamos el porcentaje de variabilidad explicada por cada componente. Solo con el PC1 se explica el 46.1%.

In [10]:
explained_variance = pca.explained_variance_ratio_

# As percentages:
explained_variance_percent = explained_variance * 100
print(explained_variance_percent)

[46.10236684 16.95890583]


Con los primeros tres componentes podemos explicar el 78.25%.

In [11]:
pca2 = PCA(n_components=3)
pca_scores = pca2.fit_transform(scaled_data)
explained_variance = pca2.explained_variance_ratio_
explained_variance_percent = explained_variance * 100
print(explained_variance_percent)
print(f'Total variance explained by PC1, PC2 and PC3: {np.sum(explained_variance_percent):.2f}%')

[46.10236684 16.95890583 15.1884362 ]
Total variance explained by PC1, PC2 and PC3: 78.25%


Y ahora graficamos por un lado las variables en función de su valor de PC1 (orden descendiente), y por el otro los paises en función de su valor PC1.
Sería esperable que los países con valores positivos de PC1 tengan valores relativamente altos de `GDP`, `Life_expect` y `Pop_growth` y valores relativamente bajos de las otras variables. Y viceversa para los países con valores negativos. Y si revisamos los datos (también se pueden ver los datos de cada país haciendo hover sobre su barra en el gráfico) podemos ver que esto se cumple.

In [12]:
pca2 = PCA(n_components=3)
pca_scores = pca2.fit_transform(scaled_data)

# Autovalores (varianzas explicadas, no en porcentaje)
eigenvalues = pca2.explained_variance_

# Obtener el autovalor de mayor varianza (PC1)
max_eigenvalue = np.max(eigenvalues)
print(f'Autovalor de mayor varianza (PC1): {max_eigenvalue:.4f}')

Autovalor de mayor varianza (PC1): 3.3467


In [13]:
loads_bars_pc1 = alt.Chart(loadings_df).mark_bar().encode(
    x=alt.X('Feature', sort='-y'),  # Sort by PC1 value in descending order
    y='PC1',
    tooltip=['Feature', 'PC1']
).properties(
    title='PCA Loadings (PC1)',
    width=600,
    height=400
)

score_bars_pc1 = alt.Chart(scores_df).mark_bar().encode(
    x=alt.X('Country', sort='-y'), # Sort countries by PC1 score
    y='PC1',
    tooltip=['Country', 'PC1'] + list(numerical_cols)
).properties(
    title='PC1 Scores by Country'
)

# Display charts side by side
(loads_bars_pc1 | score_bars_pc1)

alt.HConcatChart(...)

Idem para PC2.

In [14]:
score_bars_pc2 = alt.Chart(scores_df).mark_bar().encode(
    x=alt.X('Country', sort='-y'), # Sort countries by PC2 score
    y='PC2',
    tooltip=['Country', 'PC2']
).properties(
    title='PC2 Scores by Country'
)

loads_bars_pc2 = alt.Chart(loadings_df).mark_bar().encode(
    x=alt.X('Feature', sort='-y'),
    y='PC2',
    tooltip=['Feature', 'PC2']
).properties(
    title='PCA Loadings (PC2)',
    width=600,
    height=400
)

(loads_bars_pc2 | score_bars_pc2)

alt.HConcatChart(...)

In [17]:
# ── Comparison: sklearn PCA  vs  Oja's Rule ───────────────────────────────────
import pandas as pd
import altair as alt
import numpy as np

# ── 1. Load Oja results ───────────────────────────────────────────────────────
oja_scores   = pd.read_csv('oja_country_scores.csv')
oja_loadings = pd.read_csv('oja_pc1_loadings.csv')

# ── 2. Merge country scores ───────────────────────────────────────────────────
pca_scores_clean = scores_df[['Country', 'PC1']].rename(
    columns={'Country': 'country', 'PC1': 'pca_pc1'}
)
oja_scores_clean = oja_scores.rename(columns={'pc1_score': 'oja_pc1'})
country_cmp = pca_scores_clean.merge(oja_scores_clean, on='country')

# Align signs: flip Oja if it correlates negatively with sklearn
if np.corrcoef(country_cmp['pca_pc1'], country_cmp['oja_pc1'])[0, 1] < 0:
    country_cmp['oja_pc1']    *= -1
    oja_loadings['pc1_weight'] *= -1

# ── 3. Merge loadings ─────────────────────────────────────────────────────────
oja_loadings['feature_norm'] = oja_loadings['feature'].str.replace('.', '_', regex=False)
loadings_df['feature_norm']  = loadings_df['Feature']

loadings_cmp = (
    loadings_df[['feature_norm', 'PC1']]
    .rename(columns={'feature_norm': 'feature', 'PC1': 'pca_pc1'})
    .merge(
        oja_loadings[['feature_norm', 'pc1_weight']]
        .rename(columns={'feature_norm': 'feature', 'pc1_weight': 'oja_pc1'}),
        on='feature'
    )
)
loadings_cmp['diff'] = loadings_cmp['oja_pc1'] - loadings_cmp['pca_pc1']

# ── 4. Country scores: scatter with identity line ────────────────────────────
r_c = np.corrcoef(country_cmp['pca_pc1'], country_cmp['oja_pc1'])[0, 1]

_range = [country_cmp[['pca_pc1','oja_pc1']].min().min(),
          country_cmp[['pca_pc1','oja_pc1']].max().max()]

scatter_c = alt.Chart(country_cmp).mark_circle(size=70, opacity=0.8).encode(
    x=alt.X('pca_pc1:Q', title='sklearn PCA — PC1 score', scale=alt.Scale(domain=_range)),
    y=alt.Y('oja_pc1:Q',  title="Oja's Rule — PC1 score", scale=alt.Scale(domain=_range)),
    tooltip=['country', 'pca_pc1', 'oja_pc1']
).properties(title=f'Country scores  (r = {r_c:.4f})', width=300, height=280)

diag_c = alt.Chart(pd.DataFrame({'v': _range})).mark_line(
    color='red', strokeDash=[4,4]
).encode(x='v:Q', y='v:Q')

# ── 5. Loadings: scatter with identity line ───────────────────────────────────
r_l = np.corrcoef(loadings_cmp['pca_pc1'], loadings_cmp['oja_pc1'])[0, 1]

_lrange = [loadings_cmp[['pca_pc1','oja_pc1']].min().min() - 0.05,
           loadings_cmp[['pca_pc1','oja_pc1']].max().max() + 0.05]

scatter_l = alt.Chart(loadings_cmp).mark_circle(size=100, color='steelblue').encode(
    x=alt.X('pca_pc1:Q', title='sklearn PCA — PC1 loading', scale=alt.Scale(domain=_lrange)),
    y=alt.Y('oja_pc1:Q',  title="Oja's Rule — PC1 loading", scale=alt.Scale(domain=_lrange)),
    tooltip=['feature', 'pca_pc1', 'oja_pc1', 'diff']
).properties(title=f'Loadings  (r = {r_l:.4f})', width=300, height=280)

labels_l = alt.Chart(loadings_cmp).mark_text(dx=7, dy=-7, fontSize=9).encode(
    x='pca_pc1:Q', y='oja_pc1:Q', text='feature:N'
)

diag_l = alt.Chart(pd.DataFrame({'v': _lrange})).mark_line(
    color='red', strokeDash=[4,4]
).encode(x='v:Q', y='v:Q')

# ── 6. Country bars: two separate charts (Altair 4 compatible) ────────────────
sort_order = country_cmp.sort_values('pca_pc1', ascending=False)['country'].tolist()

bars_pca = alt.Chart(country_cmp).mark_bar(color='#4C78A8', opacity=0.85).encode(
    x=alt.X('country:N', sort=sort_order,
            axis=alt.Axis(labelAngle=-45, labelFontSize=8, title=None)),
    y=alt.Y('pca_pc1:Q', title='PC1 Score'),
    tooltip=['country', 'pca_pc1']
).properties(title='sklearn PCA', width=680, height=220)

bars_oja = alt.Chart(country_cmp).mark_bar(color='#F58518', opacity=0.85).encode(
    x=alt.X('country:N', sort=sort_order,
            axis=alt.Axis(labelAngle=-45, labelFontSize=8, title=None)),
    y=alt.Y('oja_pc1:Q', title='PC1 Score'),
    tooltip=['country', 'oja_pc1']
).properties(title="Oja's Rule", width=680, height=220)

# ── 7. Loadings: two separate bar charts ──────────────────────────────────────
load_sort = loadings_cmp.sort_values('pca_pc1', ascending=False)['feature'].tolist()

lbars_pca = alt.Chart(loadings_cmp).mark_bar(color='#4C78A8', opacity=0.85).encode(
    x=alt.X('feature:N', sort=load_sort, axis=alt.Axis(title=None)),
    y=alt.Y('pca_pc1:Q', title='PC1 Loading'),
    tooltip=['feature', 'pca_pc1']
).properties(title='sklearn PCA', width=320, height=220)

lbars_oja = alt.Chart(loadings_cmp).mark_bar(color='#F58518', opacity=0.85).encode(
    x=alt.X('feature:N', sort=load_sort, axis=alt.Axis(title=None)),
    y=alt.Y('oja_pc1:Q', title='PC1 Loading'),
    tooltip=['feature', 'oja_pc1']
).properties(title="Oja's Rule", width=320, height=220)

# ── 8. Loadings difference bar (oja − pca) ────────────────────────────────────
diff_bars = alt.Chart(loadings_cmp).mark_bar().encode(
    x=alt.X('feature:N', sort=load_sort, axis=alt.Axis(title=None)),
    y=alt.Y('diff:Q', title='Oja − PCA'),
    color=alt.condition(
        alt.datum.diff > 0,
        alt.value('#F58518'),
        alt.value('#4C78A8')
    ),
    tooltip=['feature', 'pca_pc1', 'oja_pc1', 'diff']
).properties(title='Loading difference (Oja − sklearn)', width=320, height=180)

# ── 9. Compose ────────────────────────────────────────────────────────────────
(
    ((scatter_c + diag_c) | (scatter_l + diag_l + labels_l))
    & (bars_pca & bars_oja)
    & ((lbars_pca | lbars_oja) & diff_bars)
)

alt.VConcatChart(...)